# Frozen references and causal features
Fit a local median or training-qualified daily harmonic. Inspect residuals and the four-feature profile on training history; no development labels or final observations are used.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
RUN = Path(os.environ.get('ANOMALY_RUN', str(ROOT / 'outputs' / 'v4_run'))).resolve()
CONFIG = ROOT / 'configs' / 'industry_agnostic.yaml'
from anomaly_detection.pipeline import prepare, canonical, bounds, load


In [ ]:
RUN, config = prepare(CONFIG, RUN)
start, train_end, dev_end, end = bounds(config)
data, expected, registry = canonical(RUN, train_end)
from anomaly_detection.references import References
from anomaly_detection.features import features, joint_vectors
references = References(registry).fit(data, train_end)
residuals = references.score(data)
engineered = features(residuals, registry, **config["features"])
vectors, columns = joint_vectors(engineered, list(registry))
display(pd.DataFrame(references.diagnostics))
print("Ordered features:", columns)
display(vectors.dropna().head())

In [ ]:
example = residuals.loc[residuals.entity_id.eq(residuals.entity_id.iloc[0])]
example.pivot(index="event_time", columns="metric_name", values="z").plot(figsize=(10,4), title="Frozen residuals for one entity");

MAD scaling does not establish Gaussian tail probabilities. Complete short windows are mandatory; long history is contiguous and its effective duration is saved. Historical eight-feature vectors retain recursion over the full segment. A positive scale floor is distinct from feature scaling.